In [1]:
from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import Beta, Variable, bioDraws, MonteCarlo, exp, log, Elem, bioNormalCdf
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
from biogeme.results import compile_estimation_results, calcPValue
import pandas as pd
import biogeme.database as db
import numpy as np
import biogeme.distributions as dist
import pickle
from urllib.request import urlopen
import os

In [2]:
df=pd.read_csv('final_processed_crash_dataset.csv')

C:\Users\martin.dejaeghere\AppData\Local\Temp\7\ipykernel_27484\3930622460.py:1: DtypeWarning: Columns (11,17,51,52,54,56,57,58,60,61,66,67,68,69,70,71,73,74,76,77,78,84,86,87,88,99,100,101,102,106,110,113,114,115,116,117,118,119,120,121,122,124,133) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('final_processed_crash_dataset.csv')


## Sanity check and preparing the databases

In [3]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

# Select columns that are not of type object
df_non_dummies = df[['age','severity','Number of passengers','number of involved vehicles','vma','age_2','catu', 'Num_Acc','age_opposite_mean']]



In [4]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(999)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [5]:
## First model
df_carcrashes=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_carcrashes=df_carcrashes.loc[df_carcrashes['catu'].isin([1,2])]
database_carcrashes = db.Database('database_carcrashes', df_carcrashes)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]
df_mmv=df_mmv.loc[df_mmv['severity']!=3] ## Only one value
df_mmv = df_mmv.loc[df_mmv['age_opposite_mean'] != 999].copy()

database_mmv= db.Database('database_mmv',df_mmv)


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.loc[df_pedestrian['age_opposite_mean'] != 999].copy()

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


## Variables and Betas

In [6]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)


## Model for car crashes

In [7]:
v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I
      + beta_user_category_passenger_I * user_category_passenger
      + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I_bike * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
      + beta_point_of_impact_back_I_epmd* point_of_impact_back * vehicle_e_pmd
      + beta_vehicle_type_2_light_motorized_vehicle_I* vehicle_type_2_light_motorized_vehicle
      + beta_maneuver_2_overtaking_I * maneuver_2_overtaking
      + beta_maneuver_2_without_change_of_direction_I_bike * maneuver_without_change_of_direction * vehicle_bike
      + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
      + beta_intersection_no_intersection_I * intersection_no_intersection
)

v_fatality = (  constant_F
        + beta_user_category_passenger_I * user_category_passenger
        + beta_age_F * age
        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * vehicle_type_2_large_motorized_vehicle
        + beta_vehicle_type_2_light_motorized_vehicle_F* vehicle_type_2_light_motorized_vehicle
        + beta_lighting_conditions_daylight_F * lighting_conditions_daylight
        + beta_maneuver_2_turning_right_F * maneuver_2_turning_right
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
        + beta_lighting_conditions_night_without_street_lightings_F* lighting_conditions_night_without_street_lightings
        + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [9]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_carcrashes'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_carcrashes, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = model_cst_car.estimate()



In [10]:
# Random-parameters model
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] + sigma_I*X1
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


# We integrate over B_TIME_RND using Monte-Carlo
logprob = log(MonteCarlo(prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_carcrashes,logprob,number_of_draws=10000)
model_car.modelName = "random_parameter_logit_car_crashes"

# Estimate the parameters. 
results_ml_carcrashes = model_car.estimate()


In [11]:
results_ml_carcrashes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-1.031357,0.515615,-2.000247,4.547363e-02
beta_age_F,0.040992,0.010473,3.913990,9.078354e-05
beta_gender_driver_female_I,-1.097360,0.546321,-2.008637,4.457566e-02
beta_gender_female_I,0.794670,0.172450,4.608129,4.063084e-06
beta_intersection_no_intersection_I,0.463228,0.174720,2.651267,8.019041e-03
beta_lighting_conditions_daylight_F,-0.897673,0.322410,-2.784259,5.365021e-03
beta_lighting_conditions_night_without_street_lightings_F,1.740233,0.712555,2.442244,1.459627e-02
beta_maneuver_2_overtaking_I,-0.537840,0.278532,-1.930982,5.348525e-02
beta_maneuver_2_turning_right_F,1.129073,0.350853,3.218076,1.290538e-03
beta_maneuver_2_without_change_of_direction_I_bike,0.639141,0.172049,3.714875,2.033045e-04


## MMV

In [12]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female + gender_3_male)
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + beta_point_of_impact_back_I_epmd     * point_of_impact_back * vehicle_e_pmd
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
     + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_opposite_mean
    + beta_vehicle_2_e_pmd_I           * (vehicle_2_e_pmd + vehicle_3_e_pmd)
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [13]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability1, severity)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv= model_mmv.estimate()
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.028054,0.004868,-5.762696,8.278076e-09
beta_age_I,0.033476,0.005236,6.392948,1.627174e-10
beta_gender_2_female_I,-0.902573,0.140893,-6.406085,1.493037e-10
beta_gender_female_I,1.053924,0.168224,6.264995,3.728378e-10
beta_maneuver_swerving_I,-0.508054,0.192302,-2.641959,8.242800e-03
beta_maneuver_turning_left_I,-1.283990,0.324149,-3.961107,7.460297e-05
beta_point_of_impact_back_I_bike,-1.087459,0.225257,-4.827630,1.381676e-06
beta_point_of_impact_back_I_epmd,1.135721,0.681933,1.665445,9.582405e-02
beta_surface_condition_wet_I,-0.588176,0.231118,-2.544915,1.093044e-02
beta_vehicle_2_e_pmd_I,-0.314961,0.171201,-1.839712,6.581048e-02


In [14]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [15]:
model_name = 'InitialModel_mmv'



logprob_2 = models.loglogit(U, availability1, severity)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = model_cst_mmv.estimate()



## Pedestrian

In [16]:
Beta_age_bike_4=Beta('Beta_age_bike_4',0,None,None,0)

In [17]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * intersection_no_intersection
    + beta_age_opposite_mean                             * age_opposite_mean
                                            * (number_of_involved_vehicles == 2)
    + beta_gender_2_female                   * gender_2_female
                                            * (number_of_involved_vehicles == 2)
    + beta_user_category_pedestrian          * user_category_pedestrian
    + beta_maneuver_2_turning_left           * user_category_pedestrian
                                            * (maneuver_2_turning_left + maneuver_2_turning_right)
                                            * (number_of_involved_vehicles == 2)
    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [18]:
model_name = 'ordered_probit_pedestrian'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_pedes.modelName = model_name
results_pedes = model_pedes.estimate()
results_pedes.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.014941,0.001611,9.277418,0.000000e+00
beta_age_opposite_mean,-0.007240,0.001451,-4.990185,6.032164e-07
beta_crossroad_traffic_lights,0.200828,0.077777,2.582097,9.820206e-03
beta_gender_2_female,-0.565425,0.065132,-8.681222,0.000000e+00
beta_gender_female,0.515311,0.066940,7.698048,1.376677e-14
beta_intersection_no_intersection,0.237048,0.069501,3.410725,6.479032e-04
beta_maneuver_2_turning_left,0.524868,0.111144,4.722422,2.330527e-06
beta_user_category_pedestrian,1.350974,0.071959,18.774335,0.000000e+00
tau_1,0.793729,0.106250,7.470358,7.993606e-14
tau_1_diff_2,4.169346,0.170514,24.451584,0.000000e+00


In [19]:
model_name = 'ordinal_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = model_cst_pedes.estimate()


## Single-vehicle

In [30]:
beta_user_category_passenger_mixed=beta_user_category_passenger_mean + beta_user_category_passenger_sd * bioDraws('X1', 'NORMAL')


In [31]:
utility_sv = (
    beta_age * age  +
    beta_user_category_passenger_mixed * user_category_passenger 
+ beta_long_profile_slope *long_profile_slope 
+beta_helmet_driver_yes_ebike*helmet_driver_yes*vehicle_e_bike
+ beta_number_of_passengers*user_category_driver*(number_of_passengers==1) 
)


In [32]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log(MonteCarlo(the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob,number_of_draws=10000)
model_solo_2.modelName = model_name
results_solo_2 = model_solo_2.estimate()
results_solo_2.get_estimated_parameters()

In [23]:
model_name = 'ordered_logit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = model_cst_solo.estimate()
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115696,0.064929,-32.584718,0.0
tau_1_diff_2,4.395508,0.099081,44.362708,0.0


## Likelihood-ratio test

In [24]:
def get_results(file_path):

    # Ouvrir le fichier en mode binaire
    with open(file_path, 'rb') as file:
        data = pickle.load(file)

    result = res.bioResults(data)
    
    # Retourner le résultat
    return result


#res_restricted=get_results('logit_mmv~51.pickle')
#res_unrestricted=get_results('panel_mmv~36.pickle')

#res_restricted.likelihood_ratio_test(res_unrestricted, 0.01)

## Out-of sample validation of the models

In [25]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_carcrashes)
validationData_mmv= create_validation_data(df_mmv)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [26]:
# Validate the model with the validation data for car
validation_results_car = model_car.validate(results_ml_carcrashes, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_carcrashes, validationData_car)


# Initialize variables to store log-likelihoods

loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (car)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_car += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (cars)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_car += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (cars)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





Log likelihood for 2129 validation data on car (slide 1): -347.4038355106386
Log likelihood for 1361 validation data on car (slide 2): -217.56603315237675
Log likelihood for 2129 validation data on car (constant model, slide 1): -400.5415851973453
Log likelihood for 1361 validation data on car (constant model, slide 2): -249.8457918338857
Rho-square for the validation data on car (slide 1): 0.13266475105331677
Rho-square for the validation data on car (slide 2): 0.1291987287221179


In [28]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





OSError: [Errno 22] Invalid argument: '__InitialModel_mmv_val_est_2.iter'

In [29]:

# Validate the model with the validation data for mmv
validation_results_mmv = model_pedes.validate(results_pedes, validationData_pedestrian)
validation_results_mmv_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')




OSError: [Errno 22] Invalid argument: '__ordered_probit_pedestrian_val_est_1.iter'

In [ ]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


: 